In [16]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import os 
import sys
from pathlib import Path

In [18]:
# Load scripts
from src.local_config import ROOT_W

In [19]:
PROJECT_ROOT = Path.cwd().parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = ROOT_W / "data"

In [21]:
# Open files
prices = pd.read_parquet(DATA_DIR / "prices.parquet")
sp_500_c = pd.read_parquet(DATA_DIR / "sp500_constituents.parquet")
sp_400_c = pd.read_parquet(DATA_DIR / "sp400_constituents.parquet")
all_data = pd.read_parquet(DATA_DIR / "all_data.parquet")
cheapness_scores = pd.read_parquet(DATA_DIR / "cheapness_scores.parquet")
piotrosky = pd.read_parquet(DATA_DIR / "piotrosky.parquet")
earnings_calendar = pd.read_parquet(DATA_DIR / "earnings_calendar.parquet")
short_int_trans = pd.read_parquet(DATA_DIR / "short_interest_transfo.parquet")
regime = pd.read_parquet(DATA_DIR / "regime.parquet")
gics = pd.read_parquet(DATA_DIR / "gics_info.parquet")
benchmark = pd.read_parquet(DATA_DIR / "sp500_tr.parquet")

rolling_upgrade = pd.read_csv(DATA_DIR / "rolling_scores_upgrade.csv")
rolling_downgrade = pd.read_csv(DATA_DIR / "rolling_scores_downgrade.csv")

# Cleaning the data

For the first task, our data panel should contain the key variables to help us calculate overnight ($r_{ON,t}$) and intraday ($r_{ID,t}$) returns

In [22]:
prices

,ticker,instrument_id,date,open,high,low,close,adjusted_close,volume,market_cap,status,updated
0,XEL,2,2000-01-03,19.625,19.6250,18.9375,19.0000,6.5292,2738600.0,1.271214e+09,1,2025-10-28 10:09:30
1,ED,3,2000-01-03,34.375,34.4375,33.7500,33.7500,10.3796,581900.0,7.331932e+09,1,2025-10-28 10:09:30
2,BBY,4,2000-01-03,57.750,57.8751,54.0000,57.5001,13.8958,19512431.0,1.180305e+10,1,2025-10-28 10:09:30
3,DVN,6,2000-01-03,33.125,33.5000,32.0625,32.3750,9.6808,717600.0,2.787002e+09,1,2025-10-28 10:09:30
4,CVX,12,2000-01-03,85.875,85.8750,82.5626,83.6250,15.7215,4387600.0,5.488021e+10,1,2025-10-28 10:09:30
...,...,...,...,...,...,...,...,...,...,...,...,...
8000363,AGX,1607,2024-12-31,139.270,140.6000,135.8500,137.0400,135.6920,272700.0,1.860334e+09,1,2026-03-23 06:05:42
8000364,COCO,1608,2024-12-31,36.410,37.1550,36.2710,36.9100,36.9100,303400.0,2.102465e+09,1,2026-03-25 06:04:44
8000365,ATMU,1610,2024-12-31,38.830,39.3550,38.8300,39.1800,38.9488,565200.0,3.246697e+09,1,2026-04-09 06:01:08
8000366,BNL,1611,2024-12-31,15.700,15.8900,15.6850,15.8600,14.5781,1117100.0,2.991608e+09,1,2026-04-09 06:04:23


Overnight Return
$$
r_{ON,t}=(\frac{Open_t}{Close_{t-1}})-1
$$

Intraday Return
$$
r_{ID,t}=(\frac{Close_t}{Open_{t-1}})
$$

Close-to-Close Return
$$
(1+r_{ON,t})\times(1+r_{ID,t})-1=(\frac{Close_t}{Close_{t-1}})-1
$$

In [29]:
# Ensure the date column is time set
prices['date'] = pd.to_datetime(prices['date'])

# Create the main dataframe (to combine the individual equties' returns with the benchmark return)
df = prices.copy()

"""
We should create the returns for each of the tickers by grouping the open and close prices by their ticker
1. Sort by stock and date 
2. Create the adjustment factor 
3. Adjust open and close accordingly 
4. Create the adjusted close (t-1) column (Close_{t-1})
5. Create return objects
"""

df = df.sort_values(["instrument_id", "date"])
df["adj_factor"] = df["adjusted_close"] / prices["close"]
df["adj_open"] = df["open"] * df["adj_factor"] 
df["adj_close"] = df["close"] * df["adj_factor"]

df["adj_close_t-1"] = df.groupby("instrument_id")["adj_close"].shift(1)

df["r_on"] = (df["adj_open"]/df["adj_close_t-1"]) - 1
df["r_id"] = (df["adj_close"]/df["adj_open"]) - 1
df["r_ctc"] = (1+df["r_on"]) * (1+df["r_id"]) - 1

df.dropna(inplace=True)


In [33]:
df.head()

,ticker,instrument_id,date,open,high,low,close,adjusted_close,volume,market_cap,status,updated,adj_factor,adj_open,adj_close,adj_close_t-1,r_on,r_id,r_ctc
3847967,HLT,1,2013-12-13,21.9868,22.3767,21.5468,22.0967,42.8587,3984417.0,2.175675e+10,1,2025-10-28 10:09:30,1.939597,42.645538,42.8587,41.6951,0.022795,0.004998,0.027907
3849273,HLT,1,2013-12-16,23.1266,25.9462,21.5168,21.5168,41.7339,3226908.0,2.118577e+10,1,2025-10-28 10:09:30,1.939596,44.856262,41.7339,42.8587,0.046608,-0.069608,-0.026244
3850578,HLT,1,2013-12-17,21.6068,21.9868,21.5768,21.8068,42.2963,1584449.0,2.147131e+10,1,2025-10-28 10:09:30,1.939592,41.908382,42.2963,41.7339,0.004181,0.009256,0.013476
3851884,HLT,1,2013-12-18,21.8268,22.1767,21.6568,21.8268,42.3351,1955194.0,2.149100e+10,1,2025-10-28 10:09:30,1.939593,42.335100,42.3351,42.2963,0.000917,0.000000,0.000917
3853190,HLT,1,2013-12-19,21.7468,22.4067,21.5468,22.2767,43.2078,1005007.0,2.193398e+10,1,2025-10-28 10:09:30,1.939596,42.180008,43.2078,42.3351,-0.003663,0.024367,0.020614
